<a href="https://colab.research.google.com/github/kimdonggyu2008/Personal_Study/blob/main/las_decoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class DotProductAttention(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, query, key):
        attn_weights = torch.bmm(query, key.transpose(1, 2))
        attn_weights = F.softmax(attn_weights, dim=-1)
        context = torch.bmm(attn_weights, key)
        return context, attn_weights


IGNORE_ID = -1

def pad_list(xs, pad_value):
    max_length = max([x.size(0) for x in xs])
    pad = xs[0].new_full((len(xs), max_length), pad_value)
    for i, x in enumerate(xs):
        pad[i, :x.size(0)] = x
    return pad


class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, sos_id, eos_id, hidden_size, num_layers):
        super().__init__()
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.sos_id = sos_id
        self.eos_id = eos_id
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.ModuleList()
        self.rnn += [nn.LSTMCell(embedding_dim + hidden_size, hidden_size)]
        for _ in range(1, num_layers):
            self.rnn += [nn.LSTMCell(hidden_size, hidden_size)]

        self.attention = DotProductAttention()
        self.mlp = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, vocab_size),
        )

    def zero_state(self, batch, dim):
        return torch.zeros(batch, dim)

    def forward(self, padded_input, encoder_outputs):
        ys = [y[y != IGNORE_ID] for y in padded_input]
        sos = ys[0].new([self.sos_id])
        eos = ys[0].new([self.eos_id])
        ys_in = [torch.cat([sos, y], dim=0) for y in ys]
        ys_out = [torch.cat([y, eos], dim=0) for y in ys]
        ys_in_pad = pad_list(ys_in, self.eos_id)
        ys_out_pad = pad_list(ys_out, IGNORE_ID)

        B, T = ys_in_pad.size()
        embedded = self.embedding(ys_in_pad)
        h_list = [self.zero_state(B, self.hidden_size) for _ in range(self.num_layers)]
        c_list = [self.zero_state(B, self.hidden_size) for _ in range(self.num_layers)]
        att_c = self.zero_state(B, self.hidden_size)

        y_all = []

        for t in range(T):
            rnn_input = torch.cat([embedded[:, t, :], att_c], dim=1)
            h_list[0], c_list[0] = self.rnn[0](rnn_input, (h_list[0], c_list[0]))
            for i in range(1, self.num_layers):
                h_list[i], c_list[i] = self.rnn[i](h_list[i - 1], (h_list[i], c_list[i]))
            rnn_output = h_list[-1].unsqueeze(1)
            att_c, _ = self.attention(rnn_output, encoder_outputs)
            att_c = att_c.squeeze(1)
            y_t = self.mlp(torch.cat([rnn_output.squeeze(1), att_c], dim=1))
            y_all.append(y_t)

        logits = torch.stack(y_all, dim=1)
        loss = F.cross_entropy(logits.view(B * T, -1), ys_out_pad.view(-1), ignore_index=IGNORE_ID)
        return loss


In [ ]:
vocab_size=10
embedding_dim=16
hidden_size=32
num_layers=2
sos_id=0
eos_id=9

decoder=Decoder(vocab_size,embedding_dim,sos_id,eos_id,hidden_size,num_layers)

padded_input=torch.tensor([
    [1,2,3,IGNORE_ID],
    [4,5,IGNORE_ID,IGNORE_ID]
])
encoder_outputs=torch.randon(2,5,hidden_size)

loss=decoder(padded_input,encoder_outputs)
print(loss)